# 🔄 Data Orchestration: Setting up a Parent Pipeline

## What is Data Orchestration?
In a production environment, you do not run individual pipelines manually. Orchestration is the process of aggregating multiple standalone pipelines (e.g., On-Prem ingestion, API ingestion, and SQL incremental ingestion) into a single master workflow. This allows you to schedule them all at once and set up centralized monitoring and alerting.

## ➕ Step 1: Create the Parent Pipeline
To schedule multiple pipelines together, we must group them under one roof.
1. Go to your **Author** tab and click **New pipeline**.
2. Rename the pipeline to `parent pipeline`.

## ➕ Step 2: Use the 'Execute Pipeline' Activity
The core activity for orchestration in ADF is the **Execute Pipeline** activity, which simply triggers another pipeline to run.
1. Drag the **Execute Pipeline** activity onto the canvas.
2. Rename the activity (e.g., `execute onpre`).
3. Go to the **Settings** tab.
4. Select the target child pipeline from the dropdown (e.g., your on-prem ingestion pipeline).

## ➕ Step 3: Best Practice for Passing Parameters (Crucial)
*Note on Industry Best Practices*: When your child pipeline requires parameters (like an Array of files or JSON Objects), **do not hardcode the values directly into the Execute Pipeline activity**. If you pass an Array directly, ADF will often convert it into a String, which will break your pipeline and cost you hours of debugging.

**The correct way to bypass parameters:**
1. Click on the blank canvas of your `parent pipeline` to access the pipeline-level properties.
2. Create parameters at the **Parent Pipeline level**. Ensure the names and data types (Array, Object) perfectly match the ones in your child pipeline.
3. Click back on your **Execute Pipeline** activity.
4. Under the parameters section, click **Add dynamic content**.
5. Select the parent parameter you just created (e.g., `@pipeline().parameters.files`). This safely passes the value from the parent down to the child while retaining the strict data type.

## ➕ Step 4: Add Remaining Child Pipelines
Repeat the process to bring in the rest of your data sources.
1. Drag a second **Execute Pipeline** activity and rename it for your API workflow (e.g., `execute API`).
2. Under Settings, select your `API injection` pipeline. (If it doesn't require parameters, you don't need to configure any).
3. Drag a third **Execute Pipeline** activity and set it up for your `incremental injection` (SQL to Data Lake) pipeline.

## ➕ Step 5: Preparing for Production Alerts
Orchestration isn't just about scheduling; it's about monitoring.
- In a production scenario, if a critical pipeline fails at midnight, an on-call engineer needs to be notified immediately.
- Instead of using ADF's native metrics (which can be hard to track visually in complex flows), modern architectures use **Azure Logic Apps**.
- You will connect a Web Activity to the "Failure" path (red box) of your Execute Pipeline activities to trigger a Logic App via an API call, which will instantly dispatch an email alert containing the error details.

# 🔔 Logic Apps Using Azure Data Factory: Setting up Alerts

## What is a Logic App and Why Use It?
When running automated data pipelines, you need a mechanism to notify the on-call team immediately if a critical failure occurs. Azure Data Factory (ADF) does not have a native "Send Email" activity. 
Instead, we use an **Azure Logic App**—a cloud-based web application capable of receiving HTTP requests (like a REST API endpoint) and executing subsequent actions, such as dispatching an email. ADF will essentially make an API call to the Logic App whenever a pipeline fails.

## ➕ Step 1: Create the Logic App Resource
Search for **Logic App** in the Azure Marketplace, click **Create**, and configure the resource using the following tabs:

### 1. Basics
* **Subscription & Resource Group:** Select the same Subscription and Resource Group (e.g., `ADF_Project`) where your Azure Data Factory is located.
* **Logic App Name:** Provide a globally unique name (e.g., `adf-logic-app-alert`).
* **Region:** Select the exact same region as your ADF instance.
* **Plan Type:** Choose **Workflow Service Plan (Standard)**. 
* **Zone Redundancy:** Set to **Disabled** (or leave unchecked).
  * *Explanation:* Zone redundancy duplicates your App Service Plan across different availability zones to protect against datacenter-level failures. In the video, the creator explicitly skips this to save costs for a development environment, accepting the prompt that the app "will not be zoned redundant."

### 2. Storage
* **Storage Account:** Select your existing storage account or allow Azure to create a new one. 
  * *Explanation:* Standard Logic Apps are stateful and require an Azure Storage account to save execution history.
* **Blob Service Diagnostic Settings (Post-Creation):** 
  * *Note:* This setting is not found directly inside the Logic App creation wizard. Once your Logic App provisions the Storage Account, you can navigate to that Storage Account resource, find "Diagnostic settings" under the Blob service menu, and route your blob read/write/delete logs to a Log Analytics workspace. The video skips this entirely, as standard alerts do not require blob diagnostics.

### 3. Networking
* **Enable Public Access:** Set to **Yes**.
  * *Explanation:* Your Azure Data Factory will need to communicate with this Logic App via an HTTP endpoint. Enabling public access ensures ADF can reach it.
* **Enable Virtual Network (VNet) Integration:** Set to **No**.

### 4. Monitoring
* **Enable Application Insights:** You can turn this off for this project to save on logging costs, as the Logic App's native run history is sufficient for basic email alerts.

### 5. Tags
* **Name and Value pairs (Optional):** Leave blank or add a tag like `Project : ADF_Masterclass` for billing categorization.

### 6. Review + Create
* Review your configurations to ensure there are no validation errors, then click **Create**. Once deployment is complete, click **Go to resource**.

## ➕ Step 2: Design the Logic App Workflow (The Trigger)

Once the Logic App resource is deployed, you need to define the workflow that will receive the error message from Azure Data Factory.

### 1. Create the Workflow
* Navigate to the **Workflows** tab on the left-hand menu of your Logic App resource.
* Click **Add**.
* **Workflow Name:** Enter a descriptive name (e.g., `alerts`).
* **State Type:** Select **Stateful**.
  * *Explanation:* As explained in the setup, Stateful workflows are optimized for high reliability and allow you to review the run history (inputs and outputs) of every execution. This is critical for debugging pipeline alerts. Stateless is for low latency, which is not required for sending an email.
    * **Stateful (Recommended for this project)** 
        * *What it is:* Optimized for high reliability. Stateful workflows save their execution history, inputs, and outputs to your Azure Storage account after every run. 
        * *Why we choose it:* This is critical for orchestrating data pipelines. If an alert fails to send, you can open the "Run History" and debug exactly what went wrong step-by-step. The video instructor explicitly selects this option.
    * **Stateless**
        * *What it is:* Optimized for low latency and high performance. Stateless workflows run primarily in memory and do not permanently save their run history to storage.
        * *When to use it:* Best for high-volume, real-time API scenarios where you need lightning-fast responses (e.g., thousands of requests per minute) and don't need to keep a diagnostic record of every single execution.
    * **Agent (Preview)**
        * *What it is:* A newer feature (noted as being in preview during the video) designed to integrate autonomous AI agents and intelligent routing capabilities directly into Logic App workflows.
        * *Why we skip it:* Because it is a preview feature and unnecessary for a straightforward HTTP-triggered email alert, we skip this to stick to core, production-ready data engineering practices.
* Click **Create**.

### 2. Access the Logic App Designer
* Click on your newly created `alerts` workflow.
* From the left menu inside the workflow, click on **Designer**. This opens the drag-and-drop canvas for building your logic.

### 3. Configure the HTTP Trigger
* Click **Add a trigger**.
* Search for **Request** in the search bar.
* Select the trigger named **When an HTTP request is received**.
  * *Explanation:* This trigger effectively turns your Logic App into a REST API endpoint that ADF can call using a Web Activity.

### 4. Generate the Request Schema
To process the data ADF sends, the Logic App needs to know the structure (schema) of the incoming payload.
* Click the button labeled **Use sample payload to generate schema**.
  * *Explanation:* Instead of manually writing the complex JSON schema syntax (defining objects, properties, and types), you can simply provide a mock JSON payload, and Azure will generate the schema for you.
* Paste the following pseudo-JSON block into the window:
  ```json
  {
    "pipeline_name": "blah",
    "run_id": "blah",
    "status": "blah",
    "error_message": "blah"
  }
  ```
* Click **Done**. The Logic App will automatically parse this and populate the box with the formal JSON schema, defining each variable as a string property.

### 5. Save and Generate Endpoint URL
* Click **Save** at the top of the Designer.
* *Crucial Step:* Immediately after saving, click back into the **When an HTTP request is received** trigger. You will see that an **HTTP POST URL** has now been generated. 
* **Copy this URL.** You will need to paste this exact endpoint into your Azure Data Factory Web Activity later so it knows where to send the error payload.

### 6.Exploring the Logic App Designer Features
* **Schema:** As generated in the previous step, this tab defines the exact JSON structure the HTTP trigger expects to receive. It acts as a strict contract between Azure Data Factory and your Logic App.
* **Parameters:** This section allows you to define custom variables and parameters that can be reused across multiple actions within your workflow, making your logic highly dynamic.
* **Settings:** This tab contains specific configurations for your selected trigger or action, such as concurrency controls, timeout limits, and secure inputs/outputs (useful if you are handling sensitive data).
* **Code View:** While the Designer is a visual drag-and-drop tool, you can view the underlying JSON code of your workflow. This is incredibly useful for version control, advanced debugging, or quickly copying logic between different environments.
* **About:** Provides general metadata, versioning, and documentation regarding the specific connector or trigger you are currently using.

## ➕ Step 3: Add the Email Action
1. Below your HTTP trigger, click **Add an action**.
2. Search for **Outlook** (or Office 365 Outlook) and select **Send an email (V2)**. 
   * *Note from the video:* Attempting to use the standard Gmail connector can sometimes cause authorization conflicts in this environment, so Outlook is highly recommended.
3. **Configure Connection Name & Authentication Type:**
   * **Connection Name:** Provide a clear, recognizable name for your API connection (e.g., `Outlook-Alert-Connection`). This helps you manage your API connections if you have multiple services tied to your Logic App.
   * **Authentication Type:** Select **Use default shared application** (or choose your preferred managed identity/credentials setup depending on your enterprise security policies). 
4. Sign in to authenticate your email account. This account will act as the "Sender" of the alert emails.
5. Configure the email template:
   * **To:** Enter the email address of the person or team to be notified.
   * **Subject:** Write a clear subject, e.g., `Pipeline Alert: Failure detected`.
   * **Body:** Draft a helpful message. Use the **Dynamic Content** pop-up (accessed by typing `/` or clicking the lightning bolt icon) to map the variables we defined in the trigger schema (`pipeline_name`, `run_id`, `status`, `error_message`). This ensures the on-call engineer gets the exact dynamic details of the failed run.
6. Click **Save** on the workflow. 
7. **Crucial Step:** Once saved, click on your HTTP Trigger activity again. A unique **HTTP POST URL** will now be generated. **Copy this URL**—you will need to paste this into ADF.

## ➕ Step 4: Trigger the Logic App from Azure Data Factory
Now, configure Azure Data Factory to call this Logic App whenever a pipeline fails.

1. Go back to your ADF `parent pipeline`.
2. Drag a **Web Activity** onto the canvas and rename it (e.g., `Failure Alert`).
3. Connect the **Red Arrow (On Failure)** from your target Execute Pipeline activity to this new Web Activity.
4. Go to the **Settings** tab of the Web Activity:
   * **URL:** Paste the HTTP POST URL you copied from the Logic App.
   * **Method:** Select **POST**.
   * **Body:** You must pass the actual ADF system variables mapped to the JSON schema you created. Click **Add dynamic content** and write a JSON payload utilizing ADF's expression builder:
   ```json
   {
     "pipeline_name": "@{pipeline().Pipeline}",
     "run_id": "@{pipeline().RunId}",
     "status": "@{activity('execute incremental').output.status}",
     "error_message": "@{activity('execute incremental').output.error.message}"
   }

## ➕ Step 5: Test, Debug, and Common Pitfalls
After wiring your Web Activity to trigger the Logic App, you need to properly test and publish your workflow.

1. **Initial Testing:** To test the Logic App without forcing a real failure, you can temporarily attach the Web Activity via the green **"On Success"** arrow. 
2. **The `error.message` Pitfall:** If you test on *Success*, the pipeline will throw a secondary error because `output.error.message` does not exist when an activity succeeds. 
   * *The Fix:* If you want a universal alert for both success and failure, you must write an `@if()` condition or try-catch logic in the expression builder to handle missing error objects. If wired strictly to the **On Failure** path, `output.error.message` will work perfectly.
   * **Web Activity Body Code:** In your ADF Web Activity settings, click **Add dynamic content** and paste the following JSON payload:
   ```json
      {
         "pipeline_name" : "@{pipeline().Pipeline}",
         "run_id" : "@{pipeline().RunId}",
         "status" : "@{activity('ExecuteIncremental').Status}",
         "error" : "@{if(equals(activity('ExecuteIncremental').Status, 'Failed'), string(activity('ExecuteIncremental').error), 'No error')}"
      }
3. **Run the Debug:** Click **Debug** at the top of your ADF parent pipeline canvas to execute the workflow.
4. **Check Logic App Run History:** In your Azure Logic App resource, navigate to the **Run History** tab on the left menu to verify that the HTTP request was successfully received and that the Outlook action executed.
5. **Check Spam Folder:** Automated alert emails generated by cloud applications frequently get routed to the **Spam folder**. If the Logic App Run History says "Succeeded" but you don't see the email in your primary inbox, check your Spam folder.
6. **Finalize and Publish:** Once successfully tested, ensure the Web Activity is wired exclusively to the **On Failure** path (the red arrow), and click **Publish All** in Azure Data Factory to save your workspace changes.